# Classification & Segmentation — concept importance, Elsewhere concepts, border concepts

Covers `PAPER_NOTES.md` Sections 6-7: concept-importance derivation (`φ = E(Z)W'`), the
classification-vs-segmentation task-recruitment comparison, causal verification of
"Elsewhere" concepts, and segmentation border concepts. Depth estimation stays deferred
(see `README.md`). Requires `rabbit_hull.ipynb` (Phase 1) to have been run first — this
notebook loads its cached centroids + trained SAE checkpoint.

## 0. Setup

In [ ]:
import os, sys

# Anchors on markers unique to this repo so `import config` always finds THIS
# project's src/, regardless of how/where Colab's CWD ended up (upload, git
# clone, Drive mount, ...) — sys.path.append("src") alone only works if CWD
# already happens to be the repo root, which Colab doesn't guarantee.
_MARKERS = (os.path.join("src", "config.py"), os.path.join("src", "sae.py"))
_here = os.getcwd()
_candidates = [
    _here,
    os.path.join(_here, "rabbit_hull"),
    "/content/rabbit_hull",
    os.path.dirname(_here),
]
for _root in _candidates:
    if all(os.path.isfile(os.path.join(_root, m)) for m in _MARKERS):
        os.chdir(_root)
        sys.path.insert(0, os.path.join(_root, "src"))
        break
else:
    raise RuntimeError(
        "Could not locate the repo root (needs src/config.py and src/sae.py). "
        "cd into the rabbit_hull folder first (e.g. %cd /content/rabbit_hull), "
        "then re-run this cell."
    )

print("Repo root:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from config import CONFIG
import utils
import data_utils
import ade20k_utils
import model_utils
import sae as sae_module
import probing
import rise

print(CONFIG)

## 1. Load analysis images, activations, and SAE latents

Reuses `data_utils.load_analysis_images` (the 200-class curated set) and the Phase-1
SAE checkpoint. Latents are cached *sparse* (top-k indices/values per token, not a dense
`(n, t, n_atoms)` tensor — that would be several GB at this scale) via
`sae_module.encode_sparse` / densified on demand via `sae_module.densify`.

In [ ]:
val_dataset = data_utils.load_imagenet_val(CONFIG)
analysis_images, analysis_labels = data_utils.load_analysis_images(val_dataset, CONFIG)

model, processor = model_utils.load_model(CONFIG)

analysis_acts = utils.cached(
    Path(CONFIG["cache_dir"]) / "activations" / "analysis.pt",
    lambda: model_utils.extract_activations(analysis_images, model, processor, CONFIG),
)

In [ ]:
centroids_path = Path(CONFIG["cache_dir"]) / "centroids.pt"
sae_path = Path(CONFIG["cache_dir"]) / "checkpoints" / "sae.pt"
if not (centroids_path.exists() and sae_path.exists()):
    raise RuntimeError(
        "No cached centroids/SAE checkpoint found — run rabbit_hull.ipynb (Phase 1) first."
    )
centroids = torch.load(centroids_path, weights_only=True)
sae = sae_module.load_checkpoint(centroids, CONFIG)
dictionary = sae.dictionary().detach().cpu().numpy()  # D, (n_atoms, d_model)

analysis_indices, analysis_values = utils.cached(
    Path(CONFIG["cache_dir"]) / "latents" / "analysis_sparse.pt",
    lambda: sae_module.encode_sparse(sae, analysis_acts, CONFIG),
)

## 2. Classification probe

Trained on the **cls token** per image (standard ViT image-level probing convention) —
matches `PAPER_NOTES.md` §6's `Y = A W^T` with `A` at image granularity. Labels are
remapped to a dense `0..199` range for the 200 analysis classes.

In [ ]:
unique_labels = sorted(set(analysis_labels))
label_to_idx = {lid: i for i, lid in enumerate(unique_labels)}
y_cls = np.array([label_to_idx[l] for l in analysis_labels])
X_cls = analysis_acts[:, utils.CLS_IDX, :].float().numpy()

X_train, X_val, y_train, y_val = probing.train_val_split(X_cls, y_cls, CONFIG)
cls_probe, cls_val_acc = probing.train_linear_probe(X_train, y_train, X_val, y_val, CONFIG)

## 3. Concept importance & top concepts per class

`φ = E(Z) W'` where `W' = D W^T` (§6). `Z` here is the SAE code at the cls-token position
only, since that's the population the probe above was trained/evaluated on.

In [ ]:
Z_cls = sae_module.densify(
    analysis_indices[:, utils.CLS_IDX : utils.CLS_IDX + 1, :],
    analysis_values[:, utils.CLS_IDX : utils.CLS_IDX + 1, :],
    CONFIG["n_atoms"],
).numpy().squeeze(1)  # (n_images, n_atoms)

phi_cls, contrib_cls = probing.concept_importance(Z_cls, dictionary, cls_probe.coef_)

for class_idx in range(3):
    top = probing.top_concepts_for_class(contrib_cls, class_idx, k=10)
    label_name = unique_labels[class_idx]
    print(f"Class {label_name} (idx {class_idx}) top concepts: {top.tolist()}")

## 4. Elsewhere concepts (causal verification)

For a handful of top classes, take each class's single top concept and check whether its
activation drops when the object is masked out vs. when a random same-size region is
masked out (`PAPER_NOTES.md` §3/Appendix C.2's causal claim). Since plain ImageNet
classification images have no ground-truth object mask, "the object" is approximated as
the spatial patches the classification probe's own class-weight direction scores highest
on — a self-consistent proxy (documented simplification; the paper's own method needs
either segmentation masks or a much larger RISE sweep than fits here).

Scaled down from the paper's 8000 masks/image, all classes, to
`CONFIG["rise_n_masks"]` masks x `CONFIG["rise_images_per_class"]` images x
`CONFIG["rise_n_classes"]` classes (see `README.md`).

In [ ]:
def object_region_from_probe(image_idx, class_idx, top_frac=0.25):
    """Proxy object mask: patches scoring highest against the probe's class-weight direction."""
    spatial = analysis_acts[image_idx, utils.SPATIAL_SLICE, :].float().numpy()  # (256, d_model)
    scores = spatial @ cls_probe.coef_[class_idx]
    grid = scores.reshape(16, 16)
    k = max(1, int(top_frac * grid.size))
    threshold = np.partition(scores, -k)[-k]
    return grid >= threshold


rng = np.random.RandomState(CONFIG["random_state"])
top_classes = np.argsort(-phi_cls)[: CONFIG["rise_n_classes"]]

elsewhere_results = []
for class_idx in top_classes:
    concept_idx = int(probing.top_concepts_for_class(contrib_cls, class_idx, k=1)[0])
    class_image_idx = np.where(y_cls == class_idx)[0]
    chosen = rng.choice(class_image_idx, size=min(CONFIG["rise_images_per_class"], len(class_image_idx)), replace=False)
    for image_idx in chosen:
        region = object_region_from_probe(image_idx, class_idx)
        result = rise.verify_elsewhere_concept(
            analysis_images[image_idx], region, model, processor, sae, CONFIG, concept_idx
        )
        result.update(class_idx=int(class_idx), concept_idx=concept_idx, image_idx=int(image_idx))
        elsewhere_results.append(result)
        print(
            f"class={unique_labels[class_idx]} concept={concept_idx}  "
            f"with_object={result['with_object']:.3f}  "
            f"object_masked={result['object_masked']:.3f}  "
            f"random_masked={result['random_region_masked']:.3f}"
        )

## 5. Classification task-subspace stats

Fig 3 middle/right: top-500 concepts by `φ` (aggregate across classes, via
`contrib_cls.sum(axis=1)`) vs. a random-500 control — intra-group cosine similarity and
singular-value spectrum.

In [ ]:
concept_scores_cls = contrib_cls.sum(axis=1)
top500_cls = np.argsort(-concept_scores_cls)[:500]
random500 = probing.random_control_indices(CONFIG["n_atoms"], 500, rng)

stats_cls = probing.task_subspace_stats(dictionary, top500_cls, random500)
print(f"Classification top-500 mean cosine sim: {stats_cls['task_cosine_sim']:.4f}")
print(f"Random-500 mean cosine sim: {stats_cls['random_cosine_sim']:.4f}")

plt.plot(stats_cls["task_singular_values"], label="classification top-500")
plt.plot(stats_cls["random_singular_values"], label="random-500")
plt.xlabel("component"); plt.ylabel("singular value"); plt.legend()
plt.title("Classification task subspace vs. random (Fig 3 right)")
plt.show()

## 6. Segmentation (ADE20K)

Public dataset, no HF gating. Labels are downsampled to the same 16x16 patch grid as
DINOv2's spatial tokens (`ade20k_utils.patchify_labels`), so a per-token linear probe can
be trained directly (dense prediction, unlike the image-level classification probe
above).

In [ ]:
ade_dataset = ade20k_utils.load_ade20k(CONFIG)
seg_idx = ade20k_utils.make_segmentation_split(ade_dataset, CONFIG)
seg_images = [ade_dataset[int(i)]["image"].convert("RGB") for i in seg_idx]
seg_maps = [ade_dataset[int(i)]["annotation"] for i in seg_idx]

seg_acts = utils.cached(
    Path(CONFIG["cache_dir"]) / "activations" / "ade20k.pt",
    lambda: model_utils.extract_activations(seg_images, model, processor, CONFIG),
)
patch_labels = np.stack([ade20k_utils.patchify_labels(m) for m in seg_maps])  # (n_seg, 16, 16)

seg_indices, seg_values = utils.cached(
    Path(CONFIG["cache_dir"]) / "latents" / "ade20k_sparse.pt",
    lambda: sae_module.encode_sparse(sae, seg_acts, CONFIG),
)
Z_seg_spatial = sae_module.densify(
    seg_indices[:, utils.SPATIAL_SLICE, :], seg_values[:, utils.SPATIAL_SLICE, :], CONFIG["n_atoms"]
).numpy().reshape(-1, CONFIG["n_atoms"])

In [ ]:
X_spatial = seg_acts[:, utils.SPATIAL_SLICE, :].float().numpy().reshape(-1, CONFIG["d_model"])
y_spatial = patch_labels.reshape(-1)

Xs_train, Xs_val, ys_train, ys_val = probing.train_val_split(X_spatial, y_spatial, CONFIG)
seg_probe, seg_val_acc = probing.train_linear_probe(Xs_train, ys_train, Xs_val, ys_val, CONFIG)

phi_seg, contrib_seg = probing.concept_importance(Z_seg_spatial, dictionary, seg_probe.coef_)

## 7. Border concepts

Split spatial patches by `ade20k_utils.border_mask` (label differs from a 4-connected
neighbor), compute per-group aggregate concept scores, and compare the top-100
border-associated concepts against a random control — same Fig 3 middle/right comparison
as classification, replicating the paper's "border concepts form a compact subspace"
claim.

In [ ]:
border_flat = np.stack([ade20k_utils.border_mask(patch_labels[i]) for i in range(len(patch_labels))]).reshape(-1)

_, contrib_border = probing.concept_importance(Z_seg_spatial[border_flat], dictionary, seg_probe.coef_)
border_scores = contrib_border.sum(axis=1)
top100_border = np.argsort(-border_scores)[:100]
random100 = probing.random_control_indices(CONFIG["n_atoms"], 100, rng)

stats_border = probing.task_subspace_stats(dictionary, top100_border, random100)
print(f"Border top-100 mean cosine sim: {stats_border['task_cosine_sim']:.4f}")
print(f"Random-100 mean cosine sim: {stats_border['random_cosine_sim']:.4f}")

plt.plot(stats_border["task_singular_values"], label="border top-100")
plt.plot(stats_border["random_singular_values"], label="random-100")
plt.xlabel("component"); plt.ylabel("singular value"); plt.legend()
plt.title("Segmentation border-concept subspace vs. random (Fig 3 right)")
plt.show()

## What's next

This notebook covers `PAPER_NOTES.md` §6-7 (concept importance, classification &
segmentation task recruitment, Elsewhere concepts, border concepts). Simplifications vs.
the paper, in full, are in `README.md`'s differences table — in short: RISE mask count
(300 vs. 8000/image), the probe-weight object-region proxy (no ground-truth object masks
available for plain classification images), and `n_atoms=1000` vs. the paper's 32,000
(so absolute concept counts throughout are much smaller, though the qualitative
comparisons — task subspace vs. random — should still hold directionally).

Continue with:
- `dictionary_geometry.ipynb` — token-type/footprint concepts (§8), dictionary geometry &
  co-activation baselines (§9-11).
- `mrh_analysis.ipynb` — position-embedding analysis (§12), MRH empirical evidence (§14).